# Disease ↔ Mutation Relation-Wise Merge

Merges Disease–Mutation triples from hald;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [32]:
import pandas as pd

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/DISEASE_MUTATION/ALL_DISEASE_MUTATION.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Load KG Sources

### 1a. hald

In [33]:
hald = pd.read_csv(PROC_DIR + 'hald/Disease_Mutation_HALD.csv')
hald.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"hald: {len(hald):,} rows | columns: {list(hald.columns)}")

hald['kg_type'] = 'Aging'
hald['species'] = 'Homo species'

hald.head(2)

hald: 456 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_id_is']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_id_is,kg_type,species
0,D050723,Disease_Mutation,rs2619112,Disease,associated,Mutation,HALD_KG,"Fractures, Bone",MESH,Aging,Homo species
1,DOID:9970,Disease_Mutation,rs7895833,Disease,analyzed,Mutation,HALD_KG,Obesity,DOID,Aging,Homo species


## 2. Check and Fix Duplicate Columns

In [34]:
SOURCE_DFS = [('hald', hald)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[hald] ✓ no duplicates


In [35]:
hald = hald.loc[:, ~hald.columns.duplicated(keep='first')]
print("[hald] ✓ clean")

[hald] ✓ clean


## 3. Align Schemas and Concatenate

In [36]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 456 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,D050723,Disease_Mutation,rs2619112,Disease,associated,Mutation,HALD_KG,MESH,<NA>,"Fractures, Bone",<NA>,Aging,Homo species
1,DOID:9970,Disease_Mutation,rs7895833,Disease,analyzed,Mutation,HALD_KG,DOID,<NA>,Obesity,<NA>,Aging,Homo species


## 4. Standardise Fixed-Value Columns

In [37]:
final_df['head_type']  = 'Disease'
final_df['tail_type']  = 'Mutation'
final_df['relation']   = 'Disease_Mutation'
final_df['tail_id_is'] = 'Mutation'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Disease_Mutation'}
Unique kg_source: {'HALD_KG'}


In [38]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is', 'kg_type','species']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Disease_Mutation'}
head_type           : {'Disease'}
tail_type           : {'Mutation'}
relation_type       : {'screened', 'induce', 'bear', 'diagnosed', 'induced', 'exhibit', 'noted', 'reduce', 'indicate', 'detect', 'change', 'explore', 'crossed', 'harbor', 'compare', 'driven', 'rescue', 'play', 'act', 'shared', 'encompass', 'enrolled', 'present', 'altered', 'analyzed', 'evaluated', 'impacted', 'compared', 'genotyped', 'predict', 'suggested', 'denote', 'decrease', 'caused', 'vary', 'restore', 'lead', 'shown', 'identify', 'result', 'recommended', 'modify', 'performed', 'reveal', 'progress', 'estimate', 'linked', 'characterize', 'decreased', 'influence', 'characterized', 'carry', 'associate', 'reflect', 'harbour', 'regard', 'include', 'overexpress', 'found', 'base', 'examined', 'express', 'observe', 'tested', 'observed', 'be with', 'implicated', 'disturbed', 'reported', 'remain', 'associated', 'type', 'constitute', 'farnesylate', 'reclassified', 'investigate', 'i

## 5. Deduplicate and Add Schema Columns

In [39]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")


final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 425 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,D000077277,Disease_Mutation,rs6029959,Disease,act,Mutation,HALD_KG,MESH,Mutation,Esophageal Squamous Cell Carcinoma,None,Aging,Homo species
1,D000080364,Disease_Mutation,rs704180,Disease,associated,Mutation,HALD_KG,MESH,Mutation,Multifocal Choroiditis,None,Aging,Homo species
2,D000544,Disease_Mutation,rs10137185,Disease,associated,Mutation,HALD_KG,MESH,Mutation,Alzheimer Disease,None,Aging,Homo species
3,D000544,Disease_Mutation,rs10144225,Disease,associated,Mutation,HALD_KG,MESH,Mutation,Alzheimer Disease,None,Aging,Homo species
4,D000544,Disease_Mutation,rs10197851,Disease,associated,Mutation,HALD_KG,MESH,Mutation,Alzheimer Disease,None,Aging,Homo species


## 6. QC — NaN Check

In [40]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [41]:
final_df_dedup['tail_detail_name'] = final_df_dedup['tail']

## 7. Save Output

In [42]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 425 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_MUTATION/ALL_DISEASE_MUTATION.csv


# Final mapping

In [43]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [44]:
final_file = pd.read_csv(OUT_PATH)

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='head')

final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C0279626,Disease_Mutation,rs6029959,Disease,act,Mutation,HALD_KG,MESH,Mutation,Esophageal Squamous Cell Carcinoma,rs6029959,Aging,Homo species
1,C1533060,Disease_Mutation,rs704180,Disease,associated,Mutation,HALD_KG,MESH,Mutation,Multifocal Choroiditis,rs704180,Aging,Homo species
2,C0002395,Disease_Mutation,rs10137185,Disease,associated,Mutation,HALD_KG,MESH,Mutation,Alzheimer Disease,rs10137185,Aging,Homo species
3,C0002395,Disease_Mutation,rs10144225,Disease,associated,Mutation,HALD_KG,MESH,Mutation,Alzheimer Disease,rs10144225,Aging,Homo species
4,C0002395,Disease_Mutation,rs10197851,Disease,associated,Mutation,HALD_KG,MESH,Mutation,Alzheimer Disease,rs10197851,Aging,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,DOID:9970,Disease_Mutation,rs7604576,Disease,associated,Mutation,HALD_KG,DOID,Mutation,Obesity,rs7604576,Aging,Homo species
421,DOID:9970,Disease_Mutation,rs7799039,Disease,associated,Mutation,HALD_KG,DOID,Mutation,Obesity,rs7799039,Aging,Homo species
422,DOID:9970,Disease_Mutation,rs7895833,Disease,analyzed,Mutation,HALD_KG,DOID,Mutation,Obesity,rs7895833,Aging,Homo species
423,DOID:9970,Disease_Mutation,rs987237,Disease,associated,Mutation,HALD_KG,DOID,Mutation,Obesity,rs987237,Aging,Homo species


In [45]:
final_file[final_file['head'].isna()]


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


In [46]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 425 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_MUTATION/ALL_DISEASE_MUTATION.csv


In [47]:
set(final_file['kg_type'])

{'Aging'}